# import

In [10]:
import os

import pandas as pd
import pandas.io.sql as sqlio
import psycopg2 as ps

import warnings


# Análise de negócios
## 2. Ligação com bancos de dados
Fomos contratados para formular uma análise dos preços históricos de combustíveis e retornar com uma estratégia de negócios.

Os dados para esse exercício serão retirados da [ANP](https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/serie-historica-de-precos-de-combustiveis).

Utilizaremos o Postgre17 via pgadmin.

### PostgresSQL
Criamos uma database nova chamada 'ANP' no nossos servidor PostreSQL 17.
Encoding UTF-8.

Em `Schemas` guardaremos as informações sobre nossos dados. (owner: `postgres`).

As queries estão salvas em `/queries`.

### Extract, Transform and Load (ETL - KNIME)
Faremos nossa injeção de dados pelo `KNIME`. Vamos converter todos nossos `.csv`s para `.xlsx` para facilidade de leitura.

In [2]:
def to_float(df, column):
    df[column] = df[column].replace(',', '.', regex=True)
    df[column] = df[column].astype(float)
    return df

In [3]:
def to_date(df, column):
    df[column] = pd.to_datetime(df[column], format = '%d/%m/%Y')
    return df

In [4]:
# df = pd.read_csv('local/abt/ca-2019-01.csv', sep = ';', encoding = 'ISO-8859-1')
# df = pd.read_excel('local/abt/xlsx/ca-2019-01.xlsx')

In [5]:
def convert_csv_xlsx(
    directory,
    save_to = '',
    sep : str = ';',
    encoding = 'ISO-8859-1'
    ):
    # directory = 'local/'
    files = os.listdir(directory)
    for f in files:
        path = os.path.join(directory, f)
        is_file = f.endswith('csv') and os.path.isfile(path)
        if is_file:
            try:
                print(f'\nAttempting to convert {f}')
                file_name = f.split('.')[0]
                temp = pd.read_csv(path, sep = sep, encoding = encoding)

                temp = to_float(temp, 'Valor de Venda')
                temp = to_date(temp, 'Data da Coleta')

                temp.to_excel(os.path.join(save_to, f'{file_name}.xlsx'))
            except Exception as e:
                print(f'Failed {f}')
                print(e)

In [6]:
## Only run when converting

# convert_csv_xlsx('local/abt/', save_to = 'local/abt/xlsx/')

Conectamos o KNIME a nossa base de dados utilizando os dados em `PostgreSQL 17` > `Properties`, etc

### Conectando ao banco de dados ao VS CODE

In [7]:
dbname = 'ANP'
user = 'postgres'
password = ''
with open('./local/postgres-pw') as f:
    password = f.read()
host = 'localhost'
port = '5432'


conn = ps.connect(
    dbname = dbname,
    user = user,
    password = password,
    host = host,
    port = port
)

In [11]:
sql = '''
SELECT * FROM anp.preco_combustivel
'''

In [13]:
df = sqlio.read_sql_query(sql, conn)

D:\Temp\ipykernel_33392\76855033.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = sqlio.read_sql_query(sql, conn)


## 3. Geração de Insights
### Análise exploratória de dados

In [14]:
df.head()

,regiao,estado,municipio,revenda,cnpj,nome_rua,numero_rua,complemento,bairro,cep,produto,data_coleta,valor_venda,unidade_medida,bandeira
0,SE,SP,GUARULHOS,AUTO POSTO SAKAMOTO LTDA,49.051.667/0001-02,RODOVIA PRESIDENTE DUTRA,S/N,"KM 210,5-SENT SP/RJ",BONSUCESSO,07178-580,GASOLINA,2019-01-03,4.199,R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
1,SE,SP,GUARULHOS,AUTO POSTO SAKAMOTO LTDA,49.051.667/0001-02,RODOVIA PRESIDENTE DUTRA,S/N,"KM 210,5-SENT SP/RJ",BONSUCESSO,07178-580,ETANOL,2019-01-03,2.899,R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
2,SE,SP,GUARULHOS,AUTO POSTO SAKAMOTO LTDA,49.051.667/0001-02,RODOVIA PRESIDENTE DUTRA,S/N,"KM 210,5-SENT SP/RJ",BONSUCESSO,07178-580,DIESEL S10,2019-01-03,3.349,R$ / litro,PETROBRAS DISTRIBUIDORA S.A.
3,SE,SP,GUARULHOS,AUTO POSTO SAKAMOTO LTDA,49.051.667/0001-02,RODOVIA PRESIDENTE DUTRA,S/N,"KM 210,5-SENT SP/RJ",BONSUCESSO,07178-580,GNV,2019-01-03,2.439,R$ / mÂ³,PETROBRAS DISTRIBUIDORA S.A.
4,S,RS,CANOAS,METROPOLITANO COMERCIO DE COMBUSTIVEIS LTDA,88.587.589/0001-17,AVENIDA GUILHERME SCHELL,6340,None,CENTRO,92310-000,GASOLINA,2019-01-02,4.399,R$ / litro,BRANCA


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5240904 entries, 0 to 5240903
Data columns (total 15 columns):
 #   Column          Dtype  
---  ------          -----  
 0   regiao          object 
 1   estado          object 
 2   municipio       object 
 3   revenda         object 
 4   cnpj            object 
 5   nome_rua        object 
 6   numero_rua      object 
 7   complemento     object 
 8   bairro          object 
 9   cep             object 
 10  produto         object 
 11  data_coleta     object 
 12  valor_venda     float64
 13  unidade_medida  object 
 14  bandeira        object 
dtypes: float64(1), object(14)
memory usage: 599.8+ MB


#### To datetime

In [17]:
df.data_coleta = pd.to_datetime(df.data_coleta)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5240904 entries, 0 to 5240903
Data columns (total 15 columns):
 #   Column          Dtype         
---  ------          -----         
 0   regiao          object        
 1   estado          object        
 2   municipio       object        
 3   revenda         object        
 4   cnpj            object        
 5   nome_rua        object        
 6   numero_rua      object        
 7   complemento     object        
 8   bairro          object        
 9   cep             object        
 10  produto         object        
 11  data_coleta     datetime64[ns]
 12  valor_venda     float64       
 13  unidade_medida  object        
 14  bandeira        object        
dtypes: datetime64[ns](1), float64(1), object(13)
memory usage: 599.8+ MB


#### Checando nulos

In [18]:
df.isnull().sum()

regiao                  0
estado                  0
municipio               0
revenda                 0
cnpj                    0
nome_rua                0
numero_rua           2366
complemento       4030231
bairro              13581
cep                     0
produto                 0
data_coleta             0
valor_venda             0
unidade_medida          0
bandeira                0
dtype: int64

#### Selecionando colunas relevantes

In [19]:
cols = [
    'data_coleta',
    'regiao',
    'estado',
    'municipio',
    'bandeira',
    'produto',
    'valor_venda']
df_anp = df[cols]

df_anp.head()

,data_coleta,regiao,estado,municipio,bandeira,produto,valor_venda
0,2019-01-03,SE,SP,GUARULHOS,PETROBRAS DISTRIBUIDORA S.A.,GASOLINA,4.199
1,2019-01-03,SE,SP,GUARULHOS,PETROBRAS DISTRIBUIDORA S.A.,ETANOL,2.899
2,2019-01-03,SE,SP,GUARULHOS,PETROBRAS DISTRIBUIDORA S.A.,DIESEL S10,3.349
3,2019-01-03,SE,SP,GUARULHOS,PETROBRAS DISTRIBUIDORA S.A.,GNV,2.439
4,2019-01-02,S,RS,CANOAS,BRANCA,GASOLINA,4.399


#### Ano e mes

In [20]:
df_anp['ano'] = df_anp.data_coleta.dt.year
df_anp['mes'] = df_anp.data_coleta.dt.month

df_anp.head(3)

D:\Temp\ipykernel_33392\2444541290.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_anp['ano'] = df_anp.data_coleta.dt.year
D:\Temp\ipykernel_33392\2444541290.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_anp['mes'] = df_anp.data_coleta.dt.month


,data_coleta,regiao,estado,municipio,bandeira,produto,valor_venda,ano,mes
0,2019-01-03,SE,SP,GUARULHOS,PETROBRAS DISTRIBUIDORA S.A.,GASOLINA,4.199,2019,1
1,2019-01-03,SE,SP,GUARULHOS,PETROBRAS DISTRIBUIDORA S.A.,ETANOL,2.899,2019,1
2,2019-01-03,SE,SP,GUARULHOS,PETROBRAS DISTRIBUIDORA S.A.,DIESEL S10,3.349,2019,1


In [22]:
cols = [
    'data_coleta',
    'ano',
    'mes',
    'regiao',
    'estado',
    'municipio',
    'bandeira',
    'produto',
    'valor_venda']

df_anp = df_anp[cols]

#### Estatística básica

In [26]:
df_anp[['valor_venda', 'ano']].describe().round(2)

,valor_venda,ano
count,5240904.00,5240904.00
mean,4.95,2021.51
std,1.26,1.75
min,1.80,2019.00
25%,3.89,2020.00
50%,4.94,2022.00
75%,5.94,2023.00
max,9.79,2024.00


Quais tipos de produtos são comercializados?

In [27]:
df_anp.produto.unique()

array(['GASOLINA', 'ETANOL', 'DIESEL S10', 'GNV', 'DIESEL',
       'GASOLINA ADITIVADA'], dtype=object)

Quais anos estão na base?

In [28]:
df_anp.ano.unique()

array([2019, 2020, 2021, 2022, 2023, 2024])

Estatísticas por ano

In [30]:
df_anp_anual = df_anp[['ano', 'produto', 'valor_venda']]

df_anp_anual.groupby(['produto', 'ano']).agg(['min', 'max', 'mean']).round(2)

valor_venda            
                                min   max  mean
produto            ano                         
DIESEL             2019        2.87  4.99  3.60
                   2020        2.45  5.09  3.42
                   2021        3.10  6.99  4.69
                   2022        3.14  9.00  6.63
                   2023        3.97  7.99  5.76
                   2024        4.69  8.19  5.97
DIESEL S10         2019        2.79  5.09  3.69
                   2020        2.46  5.38  3.51
                   2021        2.80  6.96  4.74
                   2022        3.59  9.65  6.73
                   2023        4.19  9.00  5.86
                   2024        4.99  8.99  6.03
ETANOL             2019        2.10  5.47  3.17
                   2020        1.80  5.15  3.18
                   2021        2.05  7.90  4.67
                   2022        2.49  7.98  4.85
                   2023        2.69  6.96  4.00
                   2024        2.63  7.35  4.07
GASOLINA           2019        3.39  6.29  4.42
                   2020        2.87  5.90  4.28
                   2021        3.10  8.00  5.89
                   2022        3.49  8.99  6.32
                   2023        4.09  8.19  5.53
                   2024        4.55  7.99  5.93
GASOLINA ADITIVADA 2020        3.69  6.00  4.59
                   2021        3.46  8.99  6.04
                   2022        4.09  9.28  6.47
                   2023        3.47  9.79  5.71
                   2024        4.69  9.79  6.12
GNV                2019        2.00  4.56  3.22
                   2020        2.00  4.78  3.17
                   2021        2.38  6.70  3.89
                   2022        3.17  7.99  5.04
                   2023        2.39  6.71  4.64
                   2024        3.54  6.89  4.77

Por estado

In [31]:
df_anp_estado = df_anp[['ano', 'produto', 'valor_venda', 'estado']]

df_anp_estado.groupby(['produto', 'ano', 'estado']).agg(['min', 'max', 'mean']).round(2)

valor_venda            
                            min   max  mean
produto ano  estado                        
DIESEL  2019 AC            4.04  4.99  4.44
             AL            3.39  4.31  3.78
             AM            3.36  4.47  3.81
             AP            3.62  4.99  4.17
             BA            3.11  4.65  3.62
...                         ...   ...   ...
GNV     2024 RN            4.29  5.99  5.10
             RS            3.99  5.94  4.72
             SC            4.29  5.75  4.96
             SE            4.80  5.19  4.93
             SP            3.58  6.19  4.52

[907 rows x 3 columns]

Quantidade de bandeiras por produto / ano

In [32]:
df_anp_bandeira = df_anp[['ano', 'produto', 'bandeira']]

df_anp_bandeira.groupby(['produto', 'ano']).bandeira.nunique()

produto             ano 
DIESEL              2019    59
                    2020    52
                    2021    57
                    2022    51
                    2023    43
                    2024    47
DIESEL S10          2019    61
                    2020    57
                    2021    57
                    2022    60
                    2023    46
                    2024    46
ETANOL              2019    63
                    2020    58
                    2021    58
                    2022    60
                    2023    47
                    2024    49
GASOLINA            2019    64
                    2020    58
                    2021    58
                    2022    60
                    2023    48
                    2024    51
GASOLINA ADITIVADA  2020    43
                    2021    57
                    2022    58
                    2023    47
                    2024    47
GNV                 2019    22
                    2020    23
              

Por estado

In [33]:
df_anp_bandeira = df_anp[['ano', 'produto', 'bandeira', 'estado']]

df_anp_bandeira.groupby(['produto', 'ano', 'estado']).bandeira.nunique()

produto  ano   estado
DIESEL   2019  AC        7
               AL        6
               AM        9
               AP        3
               BA        9
                        ..
GNV      2024  RN        6
               RS        6
               SC        7
               SE        3
               SP        7
Name: bandeira, Length: 907, dtype: int64